In [ ]:
import torch
import torchvision
from torchvision import transforms
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.callbacks.early_stopping import EarlyStoppingReason
from pytorch_lightning.loggers import TensorBoardLogger
from sklearn.metrics import precision_recall_curve
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def load_file(path):
    return np.load(path).astype(np.float32)

In [ ]:
train_transforms = transforms.Compose([
                                    transforms.ToTensor(),  # Convert numpy array to tensor
                                    transforms.Normalize([0.472], [0.301]),  # Use mean and std from preprocessing notebook
                                    transforms.RandomAffine( # Data Augmentation
                                        degrees=(-5, 5), translate=(0.05, 0.05), scale=(0.95, 1.05)),
                                        transforms.RandomResizedCrop((224, 224), scale=(0.8, 1.0))

])

val_transforms = transforms.Compose([
                                    transforms.ToTensor(),  # Convert numpy array to tensor
                                    transforms.Normalize([0.472], [0.301]),  # Use mean and std from preprocessing notebook
])

In [ ]:
# Replace YOUR_USERNAME with ur kaggle username
# Use output from part 1 of the notebooks (JPEG Reading)
train_dataset = torchvision.datasets.DatasetFolder(
    "/kaggle/input/notebooks/YOUR_USERNAME/01_data_preprocessing/train/",
    loader=load_file, extensions="npy", transform=train_transforms)

val_dataset = torchvision.datasets.DatasetFolder(
    "/kaggle/input/notebooks/YOUR_USERNAME/01_data_preprocessing/val/",
    loader=load_file, extensions="npy", transform=val_transforms)

In [ ]:
fig, axis = plt.subplots(2, 2, figsize=(9, 9))
for i in range(2):
    for j in range(2):
        random_index = np.random.randint(0, len(train_dataset))
        x_ray, label = train_dataset[random_index]
        axis[i][j].imshow(x_ray[0], cmap="bone")
        axis[i][j].set_title(f"Label:{label}")

In [ ]:
batch_size = 64#TODO
num_workers = 4# TODO

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False)

np.unique(train_dataset.targets, return_counts=True), np.unique(val_dataset.targets, return_counts=True)

In [ ]:
# Model Class
class CardiomegalyModel(pl.LightningModule):
    def __init__(self, weight=4.6):
        super().__init__()

        self.model = torchvision.models.densenet121(weights='DEFAULT')

        self.model.features.conv0 = torch.nn.Conv2d(
            1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
        )

        self.model.classifier = torch.nn.Linear(in_features=1024, out_features=1)

        self.register_buffer("pos_weight", torch.tensor([weight]))
        self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)

        self.train_acc = torchmetrics.Accuracy(task="binary", threshold=0.5)
        self.val_acc = torchmetrics.Accuracy(task="binary", threshold=0.5)

        self.train_auroc = torchmetrics.AUROC(task="binary")
        self.val_auroc = torchmetrics.AUROC(task="binary")

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.float()

        logits = self(x).squeeze(-1)
        loss = self.loss_fn(logits, y)
        preds = torch.sigmoid(logits)

        # Update metrics
        self.train_acc(preds, y.int())
        self.train_auroc(preds, y.int())

        # Log loss and metrics
        self.log("Train Loss", loss, prog_bar=True)
        self.log("Train Acc", self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("Train AUROC", self.train_auroc, on_step=False, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y = y.float()

        logits = self(x).squeeze(-1)
        loss = self.loss_fn(logits, y)
        preds = torch.sigmoid(logits)

        # Update metrics
        self.val_acc(preds, y.int())
        self.val_auroc(preds, y.int())

        # Log loss and metrics
        self.log("Val Loss", loss, prog_bar=True)
        self.log("Val Acc", self.val_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("Val AUROC", self.val_auroc, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)

In [ ]:
model = CardiomegalyModel()

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor='Val Acc',
    dirpath='/kaggle/working/densenet121',
    filename='{epoch}-{Val Loss:.2f}-{Train Loss:.2f}',
    save_top_k=3,
    mode='max')

In [ ]:
trainer = pl.Trainer(accelerator="gpu", logger=TensorBoardLogger(save_dir="./logs"), log_every_n_steps=1,
                     callbacks= checkpoint_callback,
                     max_epochs=15)

In [ ]:
trainer.fit(model, train_loader, val_loader)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

best_path = checkpoint_callback.best_model_path

model = CardiomegalyModel.load_from_checkpoint(best_path)
model.to(device)
model.eval()
model.freeze()
print(best_path)

In [ ]:
preds = []
labels = []

with torch.no_grad():
    for data, label in tqdm(val_dataset):
        data = data.to(device).float().unsqueeze(0)
        pred = torch.sigmoid(model(data)[0].cpu())
        preds.append(pred)
        labels.append(label)
preds = torch.tensor(preds)
labels = torch.tensor(labels).int()

In [ ]:
acc = torchmetrics.Accuracy(task="binary")(preds, labels)
precision = torchmetrics.Precision(task="binary")(preds, labels)
recall = torchmetrics.Recall(task="binary")(preds, labels)
cm = torchmetrics.ConfusionMatrix(num_classes=2, task="binary")(preds, labels)
cm_threshed = torchmetrics.ConfusionMatrix(num_classes=2, threshold=0.25, task="binary")(preds, labels)

print(f"Val Accuracy: {acc}")
print(f"Val Precision: {precision}")
print(f"Val Recall: {recall}")
print(f"Confusion Matrix:\n {cm}")
print(f"Confusion Matrix 2:\n {cm_threshed}")

In [ ]:
fig, axis = plt.subplots(3, 3, figsize=(9, 9))

for i in range(3):
    for j in range(3):
        rnd_idx = np.random.randint(0, len(preds))
        axis[i][j].imshow(val_dataset[rnd_idx][0][0], cmap="bone")
        axis[i][j].set_title(f"Pred:{int(preds[rnd_idx] > 0.5)}, Label:{labels[rnd_idx]}")
        axis[i][j].axis("off")

In [ ]:
model.eval()
all_probs = []
all_labels = []

device = next(model.parameters()).device

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits).squeeze()

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

y_scores = np.array(all_probs).flatten()
y_true = np.array(all_labels).flatten()

precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)

valid_idxs = np.where(recalls >= 0.95)[0]

if len(valid_idxs) == 0:
    print("No threshold achieves 95% recall")
else:
    f1_scores = 2 * (precisions[valid_idxs] * recalls[valid_idxs]) / (
        precisions[valid_idxs] + recalls[valid_idxs] + 1e-8
    )

    best_local_idx = np.argmax(f1_scores)
    best_idx = valid_idxs[best_local_idx]

    target_precision = precisions[best_idx]
    target_recall = recalls[best_idx]
    f1_at_95 = f1_scores[best_local_idx]

    print(f"At {target_recall*100:.1f}% Recall, Precision is: {target_precision:.4f}")
    print(f"F1@95 Recall: {f1_at_95:.4f}")

In [ ]:
best_threshold = thresholds[best_idx]
t = best_threshold

y_pred = (y_scores >= t).astype(int)

TP = ((y_pred == 1) & (y_true == 1)).sum()
FP = ((y_pred == 1) & (y_true == 0)).sum()
FN = ((y_pred == 0) & (y_true == 1)).sum()

precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)

print(f"Threshold: {t:.4f}")
print(f"Precision @ threshold: {precision:.4f}")
print(f"Recall @ threshold: {recall:.4f}")